# Pretrain the thermal student on public thermal data

The students have seen one building. Every verified-empty frame in v4 comes
from three scenes, none of them the lobby the rig runs in, and the thermal
student answers *person* for almost any warm human-sized shape - a lit glass
door included. Measured: that door reads 31.7 °C mean and a person 30.9 °C,
so no temperature threshold separates them, and shape is all that is left.

Two stages:

1. **Pretrain** on public thermal - people from other cameras and other
   scenes, plus thousands of *exhaustively annotated* frames with nobody in
   them. That last part is the negative evidence our own unlabelled frames
   cannot give: "the teacher found nobody" is not evidence of absence,
   "a human annotated every person and there were none" is.
2. **Fine-tune** on the rig's Celsius shards, starting from those weights.

Separate exports on purpose: our plane is absolute Celsius, these frames are
8-bit AGC intensity, and the loader refuses to mix the two scales in one
split. Only weights cross over - the normalisation statistics are
recomputed from whichever dataset is training.

> **Licence.** LLVIP and Teledyne FLIR ADAS are distributed for
> NON-COMMERCIAL research and academic use. A model pretrained on them
> inherits that question. It is recorded in every export this writes.


In [ ]:
#@title 1. Settings
DRIVE       = '/content/drive'
EXPORT_NAME = 'v4'        # the rig export to fine-tune on

# Shards converted on your own machine and uploaded (300 MB instead of the
# 11 GB of source). Set to None to download and convert here instead.
PRETRAIN_READY = DRIVE + '/MyDrive/thermal-fusion/datasets/pretrain'

FLIR_ZIP    = DRIVE + '/MyDrive/thermal-fusion/datasets/FLIR_ADAS_v2.zip'
USE_LLVIP   = False       # HuggingFace mirror, ~4 GB - only when converting here
LLVIP_REPO  = 'jsonhash/LLVIP'

PRETRAIN_EPOCHS = 25
FINETUNE_EPOCHS = 50
BATCH_SIZE, WORKERS = 32, 2

import os, sys, glob, json, time, shutil, hashlib, subprocess
REMOTE   = f'{DRIVE}/MyDrive/thermal-fusion/gexport/{EXPORT_NAME}'
DATA     = f'/content/{EXPORT_NAME}'
PRETRAIN = '/content/pretrain'
# Both checkpoint directories live on Drive: a Colab session that dies at
# epoch 40 otherwise takes the run with it, and train_students resumes from
# the resume file it leaves in --out.
PRE_OUT  = REMOTE + '/models_pretrain_public'
OUT      = REMOTE + '/models_pretrained'
print('fine-tune on', REMOTE)
print('pretrain from', PRETRAIN_READY or 'datasets downloaded here')


In [ ]:
#@title 2. Mount, check the GPU, and take the CURRENT code from the bundle
from google.colab import drive
drive.mount(DRIVE)
import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0))
assert os.path.exists(REMOTE), f'{REMOTE} not found - upload the export'

if not os.path.exists(DATA):
    os.makedirs(DATA)
    t0 = time.time()
    subprocess.run(['cp', '-a', REMOTE + '/.', DATA + '/'], check=True)
    print(f'copied the export to local disk in {time.time()-t0:.0f}s')

# The code is re-copied EVERY run, and the shards are not. A stale
# /content/v4/code is the one failure this notebook has actually hit: the
# bundle on Drive gained --init-from and the converter, the local copy from
# an earlier run did not, and the cell that needed them failed pointing at
# the wrong cause. Code is a few hundred KB; there is no reason to cache it.
shutil.rmtree(DATA + '/code', ignore_errors=True)
subprocess.run(['cp', '-a', REMOTE + '/code', DATA + '/code'], check=True)
CODE = DATA + '/code'
env = dict(os.environ); env['PYTHONPATH'] = CODE

man = json.load(open(DATA + '/manifest.json'))
sha = man.get('training_bundle', {}).get('sha256', {})
for rel, want in sha.items():
    p = os.path.join(CODE, rel)
    if rel.endswith('.ipynb') or not os.path.exists(p):
        continue
    got = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    if got != want:
        print(f'WARNING {rel}: on disk {got[:8]}, manifest says {want[:8]}')
have_init = '--init-from' in open(CODE + '/perception/train_students.py').read()
assert have_init, ('this bundle predates --init-from, so stage 2 could not '
                   'start from the pretrained weights - re-export it')
print('rig export', man['version'], '| val', man['split']['val'])


## Get the pretraining data

Either it was converted on your machine and uploaded (the fast path), or it
is downloaded and converted here. Nothing below touches the datasets when
`PRETRAIN_READY` is set.


In [ ]:
#@title 3. Pretraining shards
FLIR_ROOT = LLVIP_ROOT = None
if PRETRAIN_READY:
    assert os.path.exists(PRETRAIN_READY), f'{PRETRAIN_READY} not found'
    shutil.rmtree(PRETRAIN, ignore_errors=True)
    t0 = time.time()
    subprocess.run(['cp', '-a', PRETRAIN_READY, PRETRAIN], check=True)
    print(f'took the pre-built export in {time.time()-t0:.0f}s')
else:
    conv = CODE + '/perception/external/public_thermal.py'
    assert os.path.exists(conv), 'bundle has no converter - re-export'
    shutil.rmtree(PRETRAIN, ignore_errors=True)
    if FLIR_ZIP and os.path.exists(FLIR_ZIP):
        FLIR_ROOT = '/content/flir'
        os.makedirs(FLIR_ROOT, exist_ok=True)
        subprocess.run(['unzip', '-q', FLIR_ZIP, '-d', FLIR_ROOT], check=True)
    if USE_LLVIP:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'huggingface_hub'], check=True)
        from huggingface_hub import snapshot_download
        LLVIP_ROOT = snapshot_download(LLVIP_REPO, repo_type='dataset',
                                       local_dir='/content/llvip')

    def convert(reader, root, name, annotations=None):
        cmd = [sys.executable, '-u', '-m',
               'perception.external.public_thermal', reader,
               '--root', root, '--out', PRETRAIN, '--name', name]
        if annotations:
            cmd += ['--annotations', annotations]
        p = subprocess.run(cmd, env=env, cwd=CODE, capture_output=True,
                           text=True)
        print(p.stdout or p.stderr)
        assert p.returncode == 0, 'convert failed'

    if FLIR_ROOT:
        for c in sorted(glob.glob(FLIR_ROOT + '/**/*coco.json',
                                  recursive=True)):
            d = os.path.dirname(c)
            convert('coco', d, 'flir-' + os.path.basename(d).replace(
                    'images_', ''), annotations=c)
    if LLVIP_ROOT:
        convert('llvip', LLVIP_ROOT, 'llvip')


In [ ]:
#@title 4. What actually went in
import numpy as np
pre = json.load(open(PRETRAIN + '/manifest.json'))
states, frames, boxes = {-1: 0, 0: 0, 1: 0}, 0, 0
for f in sorted(glob.glob(PRETRAIN + '/*.npz')):
    z = np.load(f)
    s = z['thermal_label_state']
    frames += len(s); boxes += int(z['n_th_boxes'].sum())
    for k in (-1, 0, 1):
        states[k] += int((s == k).sum())
print(f'pretraining set: {frames} frames in {len(pre["sessions"])} shards')
print(f'  positive            {states[1]:6d} ({100*states[1]/frames:4.1f}%)'
      f'  - {boxes} person boxes')
print(f'  exhaustively-empty  {states[0]:6d} ({100*states[0]/frames:4.1f}%)'
      f'  <- the negatives the rig data lacks')
print(f'  unknown (cropped)   {states[-1]:6d} ({100*states[-1]/frames:4.1f}%)')
assert states[1] and states[0], ('a pretraining set needs both classes: '
                                 'with one of them the loss is minimised '
                                 'by always answering the same thing')
print('\nlicence:', pre.get('license'))


## Train

Stage one learns what a person looks like in thermal from frames this rig
will never produce. Stage two moves that onto our camera, our scenes and
absolute Celsius, starting from those weights instead of from noise.

Both write their checkpoints to Drive, so a dropped session resumes instead
of starting over.


In [ ]:
#@title 5. Stage 1 - pretrain on the public data
os.makedirs(PRE_OUT, exist_ok=True)
cmd = [sys.executable, '-u', '-m', 'perception.train_students',
       '--data', PRETRAIN, '--student', 'thermal',
       '--epochs', str(PRETRAIN_EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--workers', str(WORKERS), '--out', PRE_OUT]
print(' '.join(cmd))
p = subprocess.Popen(cmd, env=env, cwd=CODE, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
assert p.wait() == 0, 'pretrain failed'


In [ ]:
#@title 6. Stage 2 - fine-tune on the rig's shards
os.makedirs(OUT, exist_ok=True)
ckpt = PRE_OUT + '/thermal_student.pt'
assert os.path.exists(ckpt), 'stage 1 produced no checkpoint'
cmd = [sys.executable, '-u', '-m', 'perception.train_students',
       '--data', DATA, '--student', 'thermal',
       '--epochs', str(FINETUNE_EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--workers', str(WORKERS), '--out', OUT, '--init-from', ckpt]
print(' '.join(cmd))
p = subprocess.Popen(cmd, env=env, cwd=CODE, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
assert p.wait() == 0, 'fine-tune failed'


## Did it help?

Not F1, and not recall: the from-scratch model already finds 99.4% of
people. The number that is the door is **precision** and the false-positive
rate on verified-empty frames.

> Read both against the same caveat: v4's val takes all 600 of its empty
> frames from `radar3-dark-negative1`, and about 290 of those contain a
> person standing a metre from the camera. Until that session's labels are
> split, the absolute FP rate is wrong for both runs - the comparison
> between them is still fair, because both are scored against it.


In [ ]:
#@title 7. Compare, then export ONNX
def metrics_of(path):
    if not os.path.exists(path):
        return None
    c = torch.load(path, map_location='cpu', weights_only=False)
    return c.get('metrics'), c.get('manifest_version')
for label, path in [('from scratch', REMOTE + '/models/thermal_student.pt'),
                    ('pretrained  ', OUT + '/thermal_student.pt')]:
    got = metrics_of(path)
    if not got:
        print(f'{label}: no checkpoint at {path}')
        continue
    m, ver = got
    print(f"{label} ({ver}): precision {m['precision']:.3f}  "
          f"recall {m['recall']:.3f}  F1 {m['f1']:.3f}  "
          f"FP-rate {m['false_positive_rate']:.3f}")

cmd = [sys.executable, '-u', '-m', 'perception.export_students_onnx',
       '--data', REMOTE]
p = subprocess.run(cmd, env=env, cwd=CODE, capture_output=True, text=True)
print('\n' + (p.stdout or p.stderr))
print('on the Jetson:')
print('  /usr/src/tensorrt/bin/trtexec --onnx=thermal_student.onnx \\')
print('      --saveEngine=thermal_student.engine --fp16 '
      '--memPoolSize=workspace:256M')
print(f'  STUDENT_ENGINES=.../gexport/{EXPORT_NAME}/models_pretrained '
      './run_live.sh')
